In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as pl
import seaborn as sns
import sqlite3

In [2]:
purchases = pd.read_csv("../data/cleaned_customer_data.csv")

In [5]:
conn = sqlite3.connect(":memory:")

purchases.to_sql("customer_sales",conn, index=False, if_exists="replace")

407664

In [7]:
first_purchase_date = """SELECT customer_id, MIN(invoice_date) AS first_purchase_date FROM customer_sales GROUP BY customer_id"""
first_purchase_date_result = pd.read_sql(first_purchase_date, conn)
display(first_purchase_date_result)

,customer_id,first_purchase_date
0,12346.0,2009-12-14 08:34:00
1,12347.0,2010-10-31 14:20:00
2,12348.0,2010-09-27 14:59:00
3,12349.0,2010-04-29 13:20:00
4,12351.0,2010-11-29 15:23:00
...,...,...
4307,18283.0,2010-02-19 17:16:00
4308,18284.0,2010-10-04 11:33:00
4309,18285.0,2010-02-17 10:24:00
4310,18286.0,2009-12-16 10:45:00


In [17]:
NOM_first_purchase = """WITH first_purchase AS (SELECT customer_id, DATE(MIN(invoice_date), 'start of month') AS cohort_month  
                        FROM customer_sales GROUP BY customer_id),
customer_activity AS (SELECT DISTINCT customer_id, DATE(invoice_date, 'start of month') AS purchase_month FROM customer_sales)
SELECT f.cohort_month, a.purchase_month, COUNT(DISTINCT a.customer_id) AS active_customers,
((CAST(STRFTIME('%Y', a.purchase_month) AS INTEGER) - CAST(STRFTIME('%Y', f.cohort_month) AS INTEGER)) * 12
+ (CAST(STRFTIME('%m', a.purchase_month) AS INTEGER) - CAST(STRFTIME('%m', f.cohort_month) AS INTEGER)) + 1) AS cohort_index
FROM first_purchase f 
JOIN customer_activity a ON f.customer_id = a.customer_id
GROUP BY f.cohort_month, a.purchase_month
ORDER BY f.cohort_month, a.purchase_month"""

NOM_first_purchase_result = pd.read_sql(NOM_first_purchase, conn)
display(NOM_first_purchase_result)

,cohort_month,purchase_month,active_customers,cohort_index
0,2009-12-01,2009-12-01,955,1
1,2009-12-01,2010-01-01,337,2
2,2009-12-01,2010-02-01,319,3
3,2009-12-01,2010-03-01,406,4
4,2009-12-01,2010-04-01,363,5
...,...,...,...,...
86,2010-10-01,2010-11-01,97,2
87,2010-10-01,2010-12-01,35,3
88,2010-11-01,2010-11-01,325,1
89,2010-11-01,2010-12-01,35,2
